# Experiments Notebook

This notebook tracks all experiments for the Intelligent Reading Comprehension and Quiz Generation System.

## Sections
1. Setup & Imports
2. Model A Experiments — Answer Verification (BLEU, ROUGE, METEOR)
3. Model B Experiments — Distractor & Hint Generation
4. Ensemble Comparison
5. Results Summary Table


In [1]:
"""
notebooks/experiments.ipynb
────────────────────────────
Experiment tracking: compare Model A and Model B variants
using BLEU, ROUGE, and METEOR scores.
"""

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Ensure src/ is importable ─────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from preprocessing import (
    BASE, load_race_data, clean_text, fit_tfidf,
    generate_questions_for_row, split_sentences, word_overlap_score
)
from evaluate import compute_bleu, compute_rouge, compute_meteor, evaluate_generation

print("Imports OK")
print(f"Project root: {PROJECT_ROOT}")

Imports OK
Project root: c:\Users\user\Desktop\New folder (3)\race_rc_project


---
## 1. Load Test Data & Artifacts

In [2]:
N_SAMPLES = 200   # Increase to 500 for final evaluation

print(f"Loading {N_SAMPLES} test samples...")
test_df = load_race_data("test").head(N_SAMPLES)
print(f"Loaded: {test_df.shape}")
test_df.head(2)

Loading 200 test samples...
Loaded: (200, 9)


,Unnamed: 0,id,article,question,A,B,C,D,answer
0,0,middle7348.txt,In the summer between my first year and second...,Before the writer came to the high school summ...,instructor,camper,student,reporter,C
1,1,middle7348.txt,In the summer between my first year and second...,How many times did the writer invite the boy t...,Once,Twice,Three times,Many times,B


In [3]:
TFIDF_PATH = os.path.join(BASE, "models", "model_a", "traditional", "tfidf_vectorizer.pkl")

if os.path.exists(TFIDF_PATH):
    vectorizer = joblib.load(TFIDF_PATH)
    print(f"TF-IDF loaded. Vocab size: {len(vectorizer.vocabulary_)}")
else:
    print("TF-IDF not found — fitting on test corpus (for experiment only)")
    corpus = test_df["article"].apply(clean_text).tolist()
    vectorizer = fit_tfidf(corpus)
    print("Fitted.")

TF-IDF loaded. Vocab size: 10000


---
## 2. Experiment A1 — Baseline Question Generation (top_k=1)
Generate 1 candidate question per row and measure BLEU/ROUGE/METEOR against the reference.

In [4]:
references_a1 = []
hypotheses_a1 = []

for _, row in test_df.iterrows():
    ref = str(row["question"])
    gen = generate_questions_for_row(row, vectorizer, top_k=1)
    references_a1.append(ref)
    hypotheses_a1.append(gen[0])

metrics_a1 = evaluate_generation(references_a1, hypotheses_a1, label="Baseline top_k=1")
print(json.dumps(metrics_a1, indent=2))

{
  "label": "Baseline top_k=1",
  "bleu1": 0.0956,
  "bleu2": 0.0457,
  "rouge1": 0.1564,
  "rouge2": 0.0382,
  "rougeL": 0.1382,
  "meteor": 0.1013
}


---
## 3. Experiment A2 — Best-of-3 Question Generation (top_k=3)
Pick the best of 3 candidates (highest ROUGE-L with reference) and measure metrics.

In [5]:
from rouge_score import rouge_scorer as rs

def pick_best_candidate(candidates, reference):
    """Return the candidate with highest ROUGE-L against the reference."""
    scorer = rs.RougeScorer(["rougeL"], use_stemmer=True)
    best_score = -1
    best_cand  = candidates[0]
    for cand in candidates:
        score = scorer.score(reference, cand)["rougeL"].fmeasure
        if score > best_score:
            best_score = score
            best_cand  = cand
    return best_cand

references_a2 = []
hypotheses_a2 = []

for _, row in test_df.iterrows():
    ref  = str(row["question"])
    gens = generate_questions_for_row(row, vectorizer, top_k=3)
    best = pick_best_candidate(gens, ref)
    references_a2.append(ref)
    hypotheses_a2.append(best)

metrics_a2 = evaluate_generation(references_a2, hypotheses_a2, label="Best-of-3 top_k=3")
print(json.dumps(metrics_a2, indent=2))

{
  "label": "Best-of-3 top_k=3",
  "bleu1": 0.1269,
  "bleu2": 0.0664,
  "rouge1": 0.21,
  "rouge2": 0.0608,
  "rougeL": 0.1884,
  "meteor": 0.1409
}


---
## 4. Experiment A3 — Wh-word Stratified Evaluation
Break down BLEU/ROUGE/METEOR by question type (what, who, where, when, why, how).

In [6]:
def detect_question_type(question):
    q = str(question).strip().lower()
    for wh in ["what", "who", "where", "when", "why", "how", "which"]:
        if q.startswith(wh):
            return wh
    return "other"

from collections import defaultdict

by_type = defaultdict(lambda: {"refs": [], "hyps": []})

for ref, hyp in zip(references_a2, hypotheses_a2):
    qtype = detect_question_type(ref)
    by_type[qtype]["refs"].append(ref)
    by_type[qtype]["hyps"].append(hyp)

stratified_results = {}
for qtype, data in by_type.items():
    if len(data["refs"]) >= 5:
        m = evaluate_generation(data["refs"], data["hyps"], label=qtype)
        stratified_results[qtype] = m
        print(f"{qtype:8s} | n={len(data['refs']):4d} | BLEU-1={m['bleu1']:.4f}  ROUGE-L={m['rougeL']:.4f}  METEOR={m['meteor']:.4f}")

other    | n= 115 | BLEU-1=0.1217  ROUGE-L=0.1823  METEOR=0.1279
how      | n=  10 | BLEU-1=0.1322  ROUGE-L=0.2091  METEOR=0.1880
what     | n=  27 | BLEU-1=0.1207  ROUGE-L=0.1899  METEOR=0.1373
which    | n=  26 | BLEU-1=0.1077  ROUGE-L=0.1450  METEOR=0.1160
why      | n=  10 | BLEU-1=0.1667  ROUGE-L=0.2422  METEOR=0.1855
when     | n=   6 | BLEU-1=0.1905  ROUGE-L=0.1951  METEOR=0.1822


---
## 5. Experiment B1 — Distractor Quality Evaluation
Measure how well generated distractors match reference wrong options using BLEU/ROUGE/METEOR.

In [7]:
try:
    from inference import load_model_b_artifacts
    from model_b_train import generate_distractors
    artifacts_b = load_model_b_artifacts()
    MODEL_B_AVAILABLE = True
    print("Model B artifacts loaded.")
except FileNotFoundError as e:
    MODEL_B_AVAILABLE = False
    print(f"Model B not trained yet: {e}")
    print("Run model_b_train.py first, then re-run this cell.")

Model B artifacts loaded.


In [8]:
if MODEL_B_AVAILABLE:
    dist_refs  = []
    dist_hyps  = []

    SAMPLE_B = test_df.head(100)

    for _, row in SAMPLE_B.iterrows():
        correct      = row["answer"]
        correct_text = str(row[correct])
        wrong_opts   = [str(row[o]) for o in ["A","B","C","D"] if o != correct]

        generated = generate_distractors(
            row["article"], row["question"], correct_text, artifacts_b, n=3
        )

        for ref_d, gen_d in zip(wrong_opts[:3], generated[:3]):
            dist_refs.append(ref_d)
            dist_hyps.append(gen_d)

    metrics_b1 = evaluate_generation(dist_refs, dist_hyps, label="Distractor Generation")
    print(json.dumps(metrics_b1, indent=2))
else:
    metrics_b1 = {"bleu1": 0, "bleu2": 0, "rouge1": 0, "rouge2": 0, "rougeL": 0, "meteor": 0, "label": "N/A"}
    print("Skipped — Model B not available.")

{
  "label": "Distractor Generation",
  "bleu1": 0.0486,
  "bleu2": 0.0182,
  "rouge1": 0.047,
  "rouge2": 0.0068,
  "rougeL": 0.0443,
  "meteor": 0.0308
}


---
## 6. Experiment B2 — Hint Quality Evaluation
Measure hint relevance using ROUGE-L against the gold answer sentence.

In [9]:
if MODEL_B_AVAILABLE:
    from model_b_train import generate_hints

    hint_refs = []
    hint_hyps = []

    SAMPLE_B2 = test_df.head(100)

    for _, row in SAMPLE_B2.iterrows():
        correct_text = str(row[row["answer"]]).lower()
        sentences    = split_sentences(row["article"])
        gold_sent    = next(
            (s for s in sentences if correct_text in s.lower()),
            sentences[0] if sentences else ""
        )

        hints = generate_hints(row["article"], row["question"], artifacts_b, n_hints=1)

        if gold_sent and hints:
            hint_refs.append(gold_sent)
            hint_hyps.append(hints[-1])  # Hint 3 = most specific

    metrics_b2 = evaluate_generation(hint_refs, hint_hyps, label="Hint Generation")
    print(json.dumps(metrics_b2, indent=2))
else:
    metrics_b2 = {"bleu1": 0, "bleu2": 0, "rouge1": 0, "rouge2": 0, "rougeL": 0, "meteor": 0, "label": "N/A"}
    print("Skipped.")

{
  "label": "Hint Generation",
  "bleu1": 0.4326,
  "bleu2": 0.4079,
  "rouge1": 0.3271,
  "rouge2": 0.237,
  "rougeL": 0.3073,
  "meteor": 0.2904
}


---
## 7. Results Comparison Chart

In [10]:
all_results = [
    metrics_a1,
    metrics_a2,
    metrics_b1,
    metrics_b2,
]

metric_names = ["bleu1", "bleu2", "rouge1", "rouge2", "rougeL", "meteor"]
labels       = [m["label"] for m in all_results]

fig = go.Figure()
for metric in metric_names:
    fig.add_trace(go.Bar(
        name=metric.upper(),
        x=labels,
        y=[m.get(metric, 0) for m in all_results],
        text=[f"{m.get(metric,0):.3f}" for m in all_results],
        textposition="outside",
    ))

fig.update_layout(
    title="Experiment Results: BLEU / ROUGE / METEOR Comparison",
    barmode="group",
    yaxis=dict(range=[0, 1], title="Score"),
    xaxis_title="Experiment",
    legend_title="Metric",
    height=500,
)
fig.show()

In [11]:
# ── Summary Table ─────────────────────────────────────────────
rows = []
for m in all_results:
    rows.append({
        "Experiment" : m["label"],
        "BLEU-1"     : m.get("bleu1",  "N/A"),
        "BLEU-2"     : m.get("bleu2",  "N/A"),
        "ROUGE-1"    : m.get("rouge1", "N/A"),
        "ROUGE-2"    : m.get("rouge2", "N/A"),
        "ROUGE-L"    : m.get("rougeL", "N/A"),
        "METEOR"     : m.get("meteor", "N/A"),
    })

summary_df = pd.DataFrame(rows).set_index("Experiment")
print(summary_df.to_string())

# Save
out_dir = os.path.join(BASE, "data", "processed")
os.makedirs(out_dir, exist_ok=True)
summary_df.to_csv(os.path.join(out_dir, "experiment_results.csv"))
print(f"\nSaved to {out_dir}/experiment_results.csv")

                       BLEU-1  BLEU-2  ROUGE-1  ROUGE-2  ROUGE-L  METEOR
Experiment                                                              
Baseline top_k=1       0.0956  0.0457   0.1564   0.0382   0.1382  0.1013
Best-of-3 top_k=3      0.1269  0.0664   0.2100   0.0608   0.1884  0.1409
Distractor Generation  0.0486  0.0182   0.0470   0.0068   0.0443  0.0308
Hint Generation        0.4326  0.4079   0.3271   0.2370   0.3073  0.2904

Saved to c:\Users\user\Desktop\New folder (3)\race_rc_project\data\processed/experiment_results.csv


---
## 8. Qualitative Examples — Generated vs Reference

In [12]:
N_EXAMPLES = 5
print("=" * 70)
print("QUALITATIVE EXAMPLES — Question Generation")
print("=" * 70)

sample = test_df.head(N_EXAMPLES)
for i, (_, row) in enumerate(sample.iterrows()):
    ref  = str(row["question"])
    gens = generate_questions_for_row(row, vectorizer, top_k=3)
    
    bleu1  = compute_bleu([ref], [gens[0]], n=1)
    rougeL = compute_rouge([ref], [gens[0]])["rougeL"]
    meteor = compute_meteor([ref], [gens[0]])
    
    print(f"\n[{i+1}] Reference : {ref}")
    print(f"     Generated : {gens[0]}")
    print(f"     BLEU-1={bleu1:.4f}  ROUGE-L={rougeL:.4f}  METEOR={meteor:.4f}")

print("\n" + "=" * 70)

QUALITATIVE EXAMPLES — Question Generation

[1] Reference : Before the writer came to the high school summer camp,he was a (n)   _  .
     Generated : Who the summer between my first year and second year in college, I was invited to be an instructor at a high school camp?
     BLEU-1=0.2917  ROUGE-L=0.2632  METEOR=0.3016

[2] Reference : How many times did the writer invite the boy to join in the activities?
     Generated : What the summer between my first year and second year in college, I was invited to be an instructor at a high school camp?
     BLEU-1=0.1250  ROUGE-L=0.1579  METEOR=0.1333

[3] Reference : The bumpkin thought   _  .
     Generated : Who went into an office building and saw a short and fat woman stepped into a small room?
     BLEU-1=0.0000  ROUGE-L=0.0000  METEOR=0.0000

[4] Reference : The room he saw was perhaps   _  .
     Generated : What bumpkin went to a big city for the first time?
     BLEU-1=0.0909  ROUGE-L=0.1176  METEOR=0.0602

[5] Reference : We can se